In [ ]:
from dotenv import load_dotenv
from operator import itemgetter

from langchain_openai import ChatOpenAI
from langchain.chains import create_sql_query_chain
from langchain_community.utilities import SQLDatabase
from langchain_core.prompts import PromptTemplate
from langchain_community.tools.sql_database.tool import QuerySQLDataBaseTool

from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

from langchain_community.agent_toolkits import create_sql_agent


In [ ]:
load_dotenv()

In [ ]:
logging.langsmith("langchain-sql")

SQL 데이터베이스 정보 불러오기

In [ ]:
# SQLite DB에 연결
db = SQLDatabase.from_uri("sqlite:///data/finance.db")

In [ ]:
print(db.dialect)  # 데이터베이스 dialect

In [ ]:
print(db.get_usable_table_names())  # 사용 가능한 테이블 이름

LLM과 DB로 체인 생성

In [ ]:
# 옵션: 프롬프트 지정

prompt = PromptTemplate.from_template(
    """Given an input question, first create a syntactically correct {dialect} query to run, then look at the results of the query and return the answer. Unless the user specifies in his question a specific number of examples he wishes to obtain, always limit your query to at most {top_k} results. You can order the results by a relevant column to return the most interesting examples in the database.
Use the following format:

Question: "Question here"
SQLQuery: "SQL Query to run"
SQLResult: "Result of the SQLQuery"
Answer: "Final answer here"

Only use the following tables:
{table_info}

Here is the description of the columns in the tables:
`cust`: customer name
`prod`: product name
`trans`: transaction date

Question: {input}"""
).partial(dialect=db.dialect)

In [ ]:
llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)

In [ ]:
chain = create_sql_query_chain(llm, db)

In [ ]:
# 상단 프롬프트 지정 시 사용
chain = create_sql_query_chain(llm, db, prompt)

In [ ]:
generated_sql_query = chain.invoke({"question": "고객의 이름을 나열하세요"})

In [ ]:
print(generated_sql_query.__repr__())  # 생성된 쿼리를 출력합니다.

쿼리 실행

In [ ]:
query2execute = QuerySQLDataBaseTool(db=db)  # 생성한 쿼리를 실행하기 위한 도구 생성

In [ ]:
query2execute.invoke({"query": generated_sql_query})

In [ ]:
query2write = create_sql_query_chain(llm, db)  # 쿼리 생성 체인

In [ ]:
chain = query2write | query2execute  # 생성한 쿼리를 실행하기 위한 체인 생성

In [ ]:
chain.invoke({"question": "테디의 이메일을 조회하세요"})

## 답변을 LLM으로 증강-생성

In [ ]:
answer_prompt = PromptTemplate.from_template(
    """Given the following user question, corresponding SQL query, and SQL result, answer the user question.

Question: {question}
SQL Query: {query}
SQL Result: {result}
Answer: """
)

In [ ]:
answer = answer_prompt | llm | StrOutputParser()

In [ ]:
chain = (
    RunnablePassthrough.assign(query=write_query).assign(
        result=itemgetter("query") | execute_query
    )
    | answer
)

In [ ]:
chain.invoke({"question": "테디의 transaction 의 합계를 구하세요"})

## Agent

Agent로 SQL 쿼리 생성하고 실행 결과를 답변으로 출력

In [ ]:
agent_executor = create_sql_agent(llm, db=db, agent_type="openai-tools", verbose=True)

In [ ]:
agent_executor.invoke({"input": "테디와 셜리의 transaction 의 합계를 구하고 비교하세요"})